# Psalms NLP Analysis

Computational analysis of phonetic and structural properties in the Hebrew Psalms.
Generated by the Psalms NLP pipeline.

In [ ]:
import os
import sys

sys.path.insert(0, "/pipeline")

import pandas as pd
import plotly.io as pio
import psycopg2
import psycopg2.extras

pio.renderers.default = "notebook"

conn = psycopg2.connect(
    host=os.environ.get("POSTGRES_HOST", "db"),
    dbname=os.environ.get("POSTGRES_DB", "psalms"),
    user=os.environ.get("POSTGRES_USER", "psalms"),
    password=os.environ.get("POSTGRES_PASSWORD", "psalms_dev"),
)


def query(sql, params=()):
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, params)
        return [dict(r) for r in cur.fetchall()]


print("Connected.")

In [ ]:
from visualize.breath_curves import breath_curve_figure
from visualize.heatmaps import deviation_heatmap
from visualize.report import pipeline_summary_chart

# ── Deviation heatmap across all Psalms ──────────────────────────
score_rows = query("""
    SELECT v.chapter, ts.translation_key, ts.composite_deviation
    FROM translation_scores ts
    JOIN verses v ON ts.verse_id = v.verse_id
    WHERE v.book_num = 19
""")

if score_rows:
    import pandas as pd

    df = pd.DataFrame(score_rows)
    pivot = df.pivot_table(
        index="chapter",
        columns="translation_key",
        values="composite_deviation",
        aggfunc="mean",
    )
    chapters = sorted(pivot.index.tolist())
    keys = pivot.columns.tolist()
    matrix = [[float(pivot.loc[ch, k]) for k in keys] for ch in chapters]
    fig_heatmap = deviation_heatmap(
        psalm_chapters=chapters, translation_keys=keys, scores=matrix
    )
    fig_heatmap.show()

    # Save static image for Typst report
    import pathlib

    fig_dir = pathlib.Path("/data/outputs/figures")
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_heatmap.write_image(str(fig_dir / "deviation_heatmap.png"))
    print("Deviation heatmap saved.")
else:
    print("No score rows found — run Stage 4 first.")

# ── Breath curve for Psalm 23:1 ──────────────────────────────────
verse_row = query("""
    SELECT v.verse_id, bp.breath_curve
    FROM verses v
    LEFT JOIN breath_profiles bp ON bp.verse_id = v.verse_id
    WHERE v.book_num = 19 AND v.chapter = 23 AND v.verse_num = 1
""")

if verse_row and verse_row[0].get("breath_curve"):
    heb_curve = verse_row[0]["breath_curve"]
    fig_breath = breath_curve_figure(
        verse_id=verse_row[0]["verse_id"],
        hebrew_curve=heb_curve,
        translation_curves={},
        title="Psalm 23:1 — Hebrew Breath Curve",
    )
    fig_breath.show()
    fig_breath.write_image(str(fig_dir / "breath_sample.png"))
    print("Breath curve saved.")
else:
    print("No breath profile for Psalm 23:1 — run Stage 3 first.")

# ── Pipeline row count summary ───────────────────────────────────
count_rows = query("""
    SELECT 'verses' AS tbl, COUNT(*) AS cnt FROM verses WHERE book_num = 19
    UNION ALL SELECT 'word_tokens',        COUNT(*)
      FROM word_tokens wt JOIN verses v ON wt.verse_id = v.verse_id
      WHERE v.book_num = 19
    UNION ALL SELECT 'breath_profiles',    COUNT(*)
      FROM breath_profiles bp JOIN verses v ON bp.verse_id = v.verse_id
      WHERE v.book_num = 19
    UNION ALL SELECT 'translation_scores', COUNT(*)
      FROM translation_scores ts JOIN verses v ON ts.verse_id = v.verse_id
      WHERE v.book_num = 19
""")
counts = {r["tbl"]: int(r["cnt"]) for r in count_rows}
fig_summary = pipeline_summary_chart(row_counts=counts, run_history=[])
fig_summary.show()
print("Summary chart done.")